In [1]:
from dotenv import load_dotenv
from langchain_teddynote import logging

load_dotenv()
logging.langsmith("parser_practice")

LangSmith 추적을 시작합니다.
[프로젝트명]
parser_practice


In [2]:
# 캐싱
from langchain_core.globals import set_llm_cache
from langchain_community.cache import InMemoryCache

set_llm_cache(InMemoryCache())

In [3]:
# 사용자가 필요로 하는 정보를 더 명확하고 체계적인 형태로 제공할 수 있단다.
# parser랑 instruction을 뽑아보자
from langchain_core.output_parsers import PydanticOutputParser
from pydantic import BaseModel, Field

# 여기선 이메일 요약을 한다는 가정으로 만들기
class Summary(BaseModel):
    name: str = Field(description='보낸 사람의 이름')
    summ: str = Field(description='이메일을 요약한 내용')
    date: str = Field(description='이메일을 보낸 날짜')

parser = PydanticOutputParser(pydantic_object=Summary)
parser.get_format_instructions()


# 추후에 prompt에 #Question:{question} 처럼 #FORMAT: {format}이런식으로 넣어주면 됨
# 참고로 format = parser.get_format_instructions()

'The output should be formatted as a JSON instance that conforms to the JSON schema below.\n\nAs an example, for the schema {"properties": {"foo": {"title": "Foo", "description": "a list of strings", "type": "array", "items": {"type": "string"}}}, "required": ["foo"]}\nthe object {"foo": ["bar", "baz"]} is a well-formatted instance of the schema. The object {"properties": {"foo": ["bar", "baz"]}} is not well-formatted.\n\nHere is the output schema:\n```\n{"properties": {"name": {"description": "보낸 사람의 이름", "title": "Name", "type": "string"}, "summ": {"description": "이메일을 요약한 내용", "title": "Summ", "type": "string"}, "date": {"description": "이메일을 보낸 날짜", "title": "Date", "type": "string"}}, "required": ["name", "summ", "date"]}\n```'

In [ ]:
# chain이 prompt | llm 까지 일때
# output = stream_response(response, return_output=True)
output ='''어쩌구 저쩌구'''
structured_output = parser.parse(output)

In [ ]:
# 물론 chain으로 그냥 쓰면 됨
# chain = prompt | llm | parser
# response = chain.invoke(쩌구저쩌구)

In [ ]:
# 콤마 구분자 출력 파서
from langchain_classic.output_parsers import CommaSeparatedListOutputParser

output_parser = CommaSeparatedListOutputParser()

In [ ]:
# 구조화된 출력파서
from langchain_classic.output_parsers import (
    StructuredOutputParser,
    ResponseSchema
)

# 사용자의 질문에 대한 답변
response_schemas = [
    ResponseSchema(name="answer", description='사용자의 질문에 대한 답변'),
    ResponseSchema(name='source', description="사용자의 질문에 답하기 위해 사용된 '출처', '웹사이트 주소' 이어야 합니다."),
]

output_parser = StructuredOutputParser.from_response_schemas(response_schemas)



In [ ]:
format_instructions = output_parser.get_format_instructions()
"""
prompt = PromptTemplate(
....,
partial_variables={'format_instructions': format_instructions}
)
"""

In [ ]:
# JSON 출력파서
from langchain_core.output_parsers import JsonOutputParser
class Topic(BaseModel):
    description: str = Field(description='주제에 대한 간결한 설명')
    hashtags: str = Field(description='해시태그 형식의 키워드(2개 이상)')

parser = JsonOutputParser(pydantic_object=Topic)

In [ ]:
# 데이터 프레임 출력 파서
import pprint
from typing import Any, Dict
import pandas as pd
from langchain_classic.output_parsers import PandasDataFrameOutputParser

# 출력 목적
def format_parser_output(parser_output: Dict[str, Any]) -> None:
    for key in parser_output.keys():
        parser_output[key] = parser_output[key].to_dict()
    
    return pprint.PrettyPrinter(width=4, compact=True).pprint(parser_output)

df = pd.read_csv('data/titanic.csv')


In [ ]:
from langchain_core.prompts import PromptTemplate
parser = PandasDataFrameOutputParser(dataframe=df)

df_query = "Age column을 조회해 주세요."

prompt = prompt = PromptTemplate(
    template="Answer the user query.\n{format_instructions}\n{query}\n",
    input_variables=["query"],  # 입력 변수 설정
    partial_variables={
        "format_instructions": parser.get_format_instructions()
    },  # 부분 변수 설정
)

# parser_output = chain.invoke({'query':df_query})
# format_parser_output(parser_output)

In [ ]:
# 날짜 형식 출력 파서
from langchain_classic.output_parsers import DatetimeOutputParser
output_parser = DatetimeOutputParser()
output_parser.format = "%Y-%m-%d"

# prompt partial variables에 넣는다.
# 시간에 관한 질문을 한다.

In [ ]:
# 열거형 출력 파서
from langchain_classic.output_parsers import EnumOutputParser

from enum import Enum

class Colors(Enum):
    RED = "빨간색"
    GREEN = "초록색"
    BLUE = "파란색"

parser = EnumOutputParser(enum=Colors)

# 똑같이 partial variable에 넣고 색에 관한 질문을 한다.

In [ ]:
# 출력 수정 파서
from typing import List

class Actor(BaseModel):
    name: str = Field(description='name of an actor')
    film_names: List[str] = Field(
        description='list of names of films they starred in')
    
actor_query = 'Generate the filmography for a random actor'

parser = PydanticOutputParser(pydantic_object=Actor)

In [ ]:
# 잘못된 형식을 일부러 입력
misformatted = "{'name': 'Tom Hanks', 'film_names': ['Forrest Gump']}"

parser.parse(misformatted)
# 이렇게 하면 오류 출력

# import json

# misformatted = {'name': 'Tom Hanks', 'film_names': ['Forrest Gump']}

# parser.parse(json.dumps(misformatted))
# 이게 원래 올바른 입력

In [ ]:
from langchain_classic.output_parsers import OutputFixingParser

# new_parser = OutputFixingParser.from_llm(parser=parser, llm=ChatOpenAI())
# actor = new_parser.parse(misformatted)
# 이렇게 하면 actor가 제대로 나옴
# Actor(name='Tom Hanks', film_names=['Forrest Gump'])

In [ ]:
from langchain_core.output_parsers import PydanticOutputParser
from pydantic import BaseModel, Field

class emailsum(BaseModel):
    name: str = Field(description='보낸 사람의 이름')
    summ: str = Field(description='이메일 내용을 요약한 것')
    date: str = Field(description='보낸 날짜')

output_parser = PydanticOutputParser(pydantic_object=emailsum)

prompt.partial(format=output_parser.get_format_instructions())
